# SuGuard ML — обучение и честная проверка моделей

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/murzakhmetov/suguard/blob/claude/suguard-glucose-ml-model-mq4qtz/ml/notebooks/SuGuard_ML_Colab.ipynb)

Ноутбук воспроизводит всё обучение из `ml/` и добавляет то, что нельзя было сделать без доступа к открытым датасетам:

| Шаг | Что | Данные | Время (T4 GPU) |
|---|---|---|---|
| 1 | Тесты пакета | — | 1 мин |
| 2 | Модель риска диабета | NHANES 2017–2018 (реальные, HbA1c) | 2 мин |
| 3 | Планка «только глюкометр» | реальные CGM (iglu) | 1 мин |
| 4 | Нейросеть-очиститель от движения | симуляция MAX30102 | 20–40 мин |
| 5 | Пульс при **реальном** движении | PPG-DaLiA (UCI) | 15 мин |
| 6 | **Глюкоза**: проверка гипотезы на реальных данных | BIG IDEAs (PhysioNet): PPG + CGM + еда | 30–60 мин |
| 7 | Экспорт моделей и метрик | — | 1 мин |

**Как читать результаты по глюкозе.** Модель считается полезной, только если на *новых людях* (протокол B) и на *будущих днях* после калибровки (протокол C) она обгоняет простые базовые модели («предсказать среднее», «последний укол глюкометра») **и** если PPG-признаки добавляют точность сверх контекста (время суток, еда). Иначе в метаданных модели будет `validated: false`, и API покажет глюкозу только как «экспериментальную».

`Среда выполнения → Сменить среду → GPU (T4)` перед запуском.

## 0. Установка

In [ ]:
import os, sys
if not os.path.exists('suguard'):
    # Если репозиторий приватный: замените URL на https://<TOKEN>@github.com/murzakhmetov/suguard.git
    !git clone -q -b claude/suguard-glucose-ml-model-mq4qtz https://github.com/murzakhmetov/suguard.git
%cd suguard/ml
!pip install -q -r requirements.txt
import torch; print('torch', torch.__version__, 'GPU:', torch.cuda.is_available())

## 1. Тесты

In [ ]:
!python -m pytest -q

## 2. Модель риска диабета (NHANES)
Обучается на 4318 взрослых без диагноза диабета; метки — лабораторный HbA1c (≥5.7% — преддиабет/диабет, ≥6.5% — диабет). Все метрики — out-of-fold (5×5 кросс-валидация), т.е. на людях, которых модель не видела.

In [ ]:
!python scripts/train_risk.py --out models
# Survey-weighted variant (population-representative): needs the CDC demographics file
!wget -q https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/DEMO_J.xpt -O DEMO_J.xpt && python scripts/train_risk.py --out models_weighted --weights DEMO_J.xpt | tail -3
import json
r = json.load(open('models/metrics/risk_cv.json'))
for v, d in r['variants'].items():
    print(f"{v:20s}", {t: f"AUC {m['auc']:.3f} [{m['auc_95ci'][0]:.3f}-{m['auc_95ci'][1]:.3f}]" for t, m in d.items()})
print(json.dumps(r['categories_oof'], indent=1))
for t, g in r['subgroup_calibration'].items():
    for k, v in g.items(): print(f"{t:12s} {k:45s} предсказано {v['mean_predicted']:.3f}  наблюдается {v['observed']:.3f}")

## 3. Что дают одни уколы глюкометра (реальные CGM)
Планка для любой неинвазивной модели: точность «предсказания» глюкозы по последним калибровочным уколам, без браслета.

In [ ]:
!python scripts/benchmark_calibration.py --out models/metrics

## 4. Нейросеть-очиститель сигнала от движения
Генерация 150 тыс. окон симулятора MAX30102 (на CPU Colab ~25 мин; можно уменьшить `--train`) и обучение на GPU.

In [ ]:
!python scripts/make_denoiser_dataset.py --out data/denoiser --train 150000 --val 4000 --test 4000 --workers 2
!python scripts/train_denoiser.py --data data/denoiser --steps 12000 --bs 128 --out models
!python scripts/eval_denoiser.py --data data/denoiser --out models/metrics

## 5. Пульс при реальном движении: PPG-DaLiA
15 человек, ~2.5 ч бытовой активности, эталонный пульс по ЭКГ. Сигнал E4 — один зелёный канал, поэтому двухволновая логика нейросети тут не работает в полную силу (честная проверка «в худших условиях»).

In [ ]:
if not os.path.exists('PPG_FieldStudy'):
    !wget -q https://archive.ics.uci.edu/static/public/495/ppg+dalia.zip -O dalia.zip && unzip -q -o dalia.zip
    !unzip -q -o PPG_FieldStudy.zip 2>/dev/null || true
!python scripts/eval_dalia.py --root PPG_FieldStudy --out models/metrics

## 6. Глюкоза: проверка на реальных данных BIG IDEAs
16 взрослых (норма/преддиабет), браслет Empatica E4 (PPG 64 Гц) + Dexcom G6 CGM 10 дней + дневник питания. Скачивание ~ несколько ГБ с зеркала AWS Open Data (без аккаунта).

In [ ]:
!pip install -q awscli
!aws s3 sync --no-sign-request --only-show-errors s3://physionet-open/big-ideas-glycemic-wearable/1.1.2/ big-ideas
# запасной вариант (медленнее):
# !wget -r -N -c -np -nH --cut-dirs=3 -P big-ideas https://physionet.org/files/big-ideas-glycemic-wearable/1.1.2/
!ls big-ideas | head

In [ ]:
!python scripts/build_bigideas_table.py --root big-ideas --out data/bigideas_paired.parquet --every-n 3
!python scripts/train_glucose.py --table data/bigideas_paired.parquet --out models/bigideas --name bigideas

In [ ]:
p = 'models/bigideas/metrics/bigideas_eval.json'
rep = json.load(open(p))
def row(name, m):
    if not m: return
    r2 = m['r2'] if m['r2'] is None else round(m['r2'], 2)
    print(f"{name:48s} MARD {m['mard_pct']:5.1f}%  MAE {m['mae_mgdl']:5.1f}  R2 {r2}  Кларк A+B {m['clarke_ab_pct']:5.1f}%")
row('A: случайное разбиение (УТЕЧКА, завышено)', rep['A_random_rows_LEAKY'])
row('B: новые люди — модель', rep['B_subject_wise']['model'])
row('B: новые люди — «среднее»', rep['B_subject_wise']['baseline_train_mean'])
c = rep['C_personalised']
if c:
    row('C: модель + калибровка', c['model_plus_calibration'])
    for k in ('calibration_only_kalman', 'last_fingerstick_carried_forward', 'personal_time_of_day_profile'):
        row('C: ' + k, c[k])
for k, v in rep['controls'].items():
    row(f'контроль {k} (B)', v['B']); row(f'контроль {k} (C)', v['C'])
print('выигрыш:', json.dumps(rep['relative_mard_gain'], indent=1))
print('validated =', rep['validated'])
print('причины:', *rep['not_validated_because'], sep='\n  - ')
# Для BIG IDEAs validated всегда False: это не данные браслета SuGuard и всего 16 человек.
# Смотрите на выигрыш над базовыми моделями и контролями: он показывает, есть ли в PPG информация о глюкозе.

## 7. Экспорт
Скачайте архив и пришлите его (или `models/metrics/*.json`) — по метрикам решаем, можно ли показывать глюкозу пользователям.

In [ ]:
!zip -q -r suguard_ml_models.zip models
from google.colab import files
files.download('suguard_ml_models.zip')